# Violence Detection in Videos
### CNN + LSTM Deep Learning Pipeline
**Author:** Nikita Saini | IIIT Kota

Binary video classification: **Violence** vs **NonViolence**

**Architecture:** TimeDistributed CNN (spatial features) → LSTM (temporal sequence) → Dense

**Dataset:** Real Life Violence Situations Dataset

## 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    LSTM, Dense, Dropout,
    TimeDistributed, BatchNormalization,
    GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "loglevel;quiet"
print("All imports successful ✅")

## 2. Configuration

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
IMG_SIZE        = 64       # frame resize (64x64)
SEQUENCE_LENGTH = 8        # frames per video
BATCH_SIZE      = 4
EPOCHS          = 30
LEARNING_RATE   = 0.0003
TEST_SIZE       = 0.2      # 80-20 train-test split
RANDOM_STATE    = 42

CLASSES = ["NonViolence", "Violence"]

# ── Dataset path (update this) ───────────────────────────────
DATASET_PATH = "dataset/"  # update this path

print(f"Config loaded — IMG: {IMG_SIZE}x{IMG_SIZE}, Seq: {SEQUENCE_LENGTH}, Classes: {CLASSES}")

## 3. Frame Extraction

In [ ]:
def extract_frames(video_path, img_size=IMG_SIZE, seq_len=SEQUENCE_LENGTH):
    """
    Extracts `seq_len` evenly-spaced frames from a video.
    Returns: np.array of shape (seq_len, img_size, img_size, 3), or None on failure.
    """
    frames = []
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    # evenly space frame indices across the full video
    frame_indices = np.linspace(0, total_frames - 1, seq_len, dtype=int)
    frame_set = set(frame_indices)

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count in frame_set:
            try:
                frame = cv2.resize(frame, (img_size, img_size))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame.astype("float32") / 255.0
                frames.append(frame)
            except Exception:
                cap.release()
                return None

        frame_count += 1

    cap.release()

    if len(frames) == 0:
        return None

    # pad with last frame if short
    while len(frames) < seq_len:
        frames.append(frames[-1])

    return np.array(frames[:seq_len])

## 4. Load Dataset

In [ ]:
X, y = [], []
skipped = 0

for label, cls in enumerate(CLASSES):
DATASET_PATH = "dataset/"  # update this path

    if not os.path.exists(folder):
        print(f"❌ Folder not found: {folder}")
        continue

    files = [f for f in os.listdir(folder)
             if f.lower().endswith((".mp4", ".avi", ".mov"))]

    print(f"📂 {cls}: {len(files)} videos found")

    for file in files:
        video_path = os.path.join(folder, file)
        frames = extract_frames(video_path)

        if frames is not None:
            X.append(frames)
            y.append(label)
        else:
            skipped += 1

print(f"\n✅ Loaded: {len(X)} videos | Skipped: {skipped}")
print(f"Class distribution — NonViolence: {y.count(0)}, Violence: {y.count(1)}")

## 5. Preprocessing & Train-Test Split

In [ ]:
X = np.array(X, dtype='float16')   # float16 saves memory
y_raw = np.array(y)
y_cat = to_categorical(y_raw, num_classes=len(CLASSES))

print("Dataset shape:", X.shape)   # (N, 8, 64, 64, 3)

# Stratified split — keeps class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_raw         # ensures equal class ratio
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 6. Model — TimeDistributed CNN + LSTM

In [ ]:
def build_model(seq_len, img_size, num_classes, lr):
    """
    Architecture:
      TimeDistributed CNN (3 blocks) → extracts spatial features per frame
      GlobalAveragePooling2D         → reduces spatial dims
      LSTM(64)                       → learns temporal patterns across frames
      Dense(32) → Dense(num_classes) → classification
    """
    model = Sequential([
        # Block 1
        TimeDistributed(Conv2D(32, (3,3), activation='relu', padding='same'),
                        input_shape=(seq_len, img_size, img_size, 3)),
        TimeDistributed(BatchNormalization()),
        TimeDistributed(MaxPooling2D((2,2))),

        # Block 2
        TimeDistributed(Conv2D(64, (3,3), activation='relu', padding='same')),
        TimeDistributed(BatchNormalization()),
        TimeDistributed(MaxPooling2D((2,2))),

        # Block 3
        TimeDistributed(Conv2D(128, (3,3), activation='relu', padding='same')),
        TimeDistributed(BatchNormalization()),
        TimeDistributed(MaxPooling2D((2,2))),

        # Spatial pooling
        TimeDistributed(GlobalAveragePooling2D()),

        # Temporal learning
        LSTM(64, return_sequences=False),
        Dropout(0.5),

        # Classification head
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=lr),
        metrics=['accuracy']
    )
    return model

model = build_model(SEQUENCE_LENGTH, IMG_SIZE, len(CLASSES), LEARNING_RATE)
model.summary()

## 7. Training

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),   # proper held-out validation
    callbacks=callbacks,
    verbose=1
)

model.save('violence_detection_model.h5')
print("\n✅ Model saved: violence_detection_model.h5")

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2, linestyle='--')
axes[0].set_title('Accuracy Curve', fontsize=13)
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--')
axes[1].set_title('Loss Curve', fontsize=13)
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Violence Detection Model — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 9. Evaluation on Test Set

In [ ]:
# Evaluate on proper held-out test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

# Predictions
y_pred_probs  = model.predict(X_test)
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

# Classification Report
print("\n" + "="*50)
print("Classification Report")
print("="*50)
print(classification_report(y_true_labels, y_pred_labels, target_names=CLASSES))

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true_labels, y_pred_labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    linewidths=0.5
)
plt.title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 11. Predict on a Single Video

In [ ]:
def predict_video(video_path, model, classes=CLASSES):
    """
    Predicts class of a single video with confidence score.
    """
    frames = extract_frames(video_path)

    if frames is None:
        print("❌ Could not load video.")
        return

    # Visualise extracted frames
    plt.figure(figsize=(14, 3))
    for i, frame in enumerate(frames):
        plt.subplot(1, len(frames), i + 1)
        plt.imshow(frame)
        plt.axis('off')
        plt.title(f'F{i+1}', fontsize=8)
    plt.suptitle('Extracted Frames', fontsize=11)
    plt.tight_layout()
    plt.show()

    # Predict
    input_tensor = np.expand_dims(frames, axis=0).astype('float32')
    probs = model.predict(input_tensor, verbose=0)[0]
    predicted_idx = np.argmax(probs)
    confidence = probs[predicted_idx] * 100

    print(f"\n{'='*40}")
    for cls, prob in zip(classes, probs):
        print(f"  {cls:15s}: {prob*100:.2f}%")
    print(f"{'='*40}")
    print(f"  Prediction : {classes[predicted_idx]}")
    print(f"  Confidence : {confidence:.2f}%")
    print(f"{'='*40}\n")


# ── Test on a sample video (update path) ─────────────────────
sample_video = "path/to/test_video.mp4"  # update this path
predict_video(sample_video, model)